# Train From Saved Self-Play

Notebook này chạy `train_from_self_play.py` để huấn luyện model từ các file `.pt` đã được tạo bởi self-play.

In [ ]:
from pathlib import Path
import shlex
import subprocess


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "legacy" / "AlphaZero-based-autonomous-driving").is_dir() and (candidate / "source" / "highway-env").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root. Open the notebook from inside this repository.")


def run_command(command, cwd: Path):
    print("$ " + " ".join(shlex.quote(str(part)) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")


repo_root = find_repo_root()
package_root = repo_root / "legacy" / "AlphaZero-based-autonomous-driving"
runner = ["uv", "run", "python", "-m"]
train_module = "AlphaZero.scripts.train_from_self_play"
print(f"repo_root={repo_root}")
print(f"package_root={package_root}")
print(f"train_module={train_module}")

In [ ]:
input_dir = repo_root / "legacy" / "AlphaZero-based-autonomous-driving" / "outputs" / "self_play_parallel"
model_out = repo_root / "legacy" / "AlphaZero-based-autonomous-driving" / "outputs" / "alphazero_from_self_play.pth"
metrics_out = model_out.with_suffix(model_out.suffix + ".metrics.json")

config = {
    "input_dir": input_dir,
    "pattern": "*.pt",
    "recursive": False,
    "limit_files": None,
    "model_in": None,
    "model_out": model_out,
    "metrics_out": metrics_out,
    "n_residual_layers": 10,
    "batch_size": 32,
    "epochs": 10,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "device": "auto",
    "shuffle": True,
}

config

In [ ]:
command = [
    *runner,
    train_module,
    "--input-dir", str(config["input_dir"]),
    "--pattern", str(config["pattern"]),
    "--model-out", str(config["model_out"]),
    "--metrics-out", str(config["metrics_out"]),
    "--n-residual-layers", str(config["n_residual_layers"]),
    "--batch-size", str(config["batch_size"]),
    "--epochs", str(config["epochs"]),
    "--learning-rate", str(config["learning_rate"]),
    "--weight-decay", str(config["weight_decay"]),
    "--device", str(config["device"]),
]

if config["recursive"]:
    command.append("--recursive")
if config["limit_files"] is not None:
    command.extend(["--limit-files", str(config["limit_files"])])
if config["model_in"] is not None:
    command.extend(["--model-in", str(config["model_in"])])
if not config["shuffle"]:
    command.append("--no-shuffle")

run_command(command, cwd=package_root)